In [1]:
# Parameters
DB_PATH          = "../../../DB/oedb_refiner_1st.db"
BENCHMARK_PATH   = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET  = "questions"
MATCHED_CSV_PATH = "matched_questions.csv"
NOTEGROUP_ID_MIN = 1
NOTEGROUP_ID_MAX = 23
MATCH_THRESHOLD  = 70

WEIGHTS = {
    "question_content":          98,
    "main_indicator":             1,
    "followed_question_content":  1,
}


In [2]:
import sqlite3
import pandas as pd
from rapidfuzz import fuzz
import re

pd.reset_option("display.max_rows")
pd.reset_option("display.max_colwidth")

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT q.questionID, q.notegroupID, q.question_content,
                  q.main_indicator, q.followed_questionID,
                  p.question_content AS followed_question_content
           FROM questions q
           LEFT JOIN questions p ON q.followed_questionID = p.questionID
           WHERE q.notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    return df

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    # self-join to resolve followed_question_content
    fq = df[["questionID", "question_content"]].rename(columns={
        "questionID":       "followed_questionID",
        "question_content": "followed_question_content"
    })
    df = df.merge(fq, on="followed_questionID", how="left")
    return df

etl = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm  = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

print("ETL records:      ", len(etl))
print("Benchmark records:", len(bm))

ETL records:       416
Benchmark records: 430


In [3]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)  # normalise whitespace around newlines
    return s

def question_pair_score(etl_row, bm_row):
    total_weight = 0
    weighted_sum = 0.0
    for field, weight in WEIGHTS.items():
        e = normalise_str(etl_row.get(field))
        b = normalise_str(bm_row.get(field))
        if e is None or b is None:
            continue   # exclude null fields from weighted average
        sim = fuzz.ratio(e, b)
        weighted_sum += sim * weight
        total_weight += weight
    if total_weight == 0:
        return 0.0
    return weighted_sum / total_weight

In [4]:
def substring_position(etl_row, bm_row, threshold):
    """
    If BM question_content (or ETL) is a substring of the other via partial_ratio,
    return the position (0-1) of the match within the longer string — later = higher.
    Returns -1 if no substring match.
    """
    e = normalise_str(etl_row.get("question_content"))
    b = normalise_str(bm_row.get("question_content"))
    if e is None or b is None:
        return -1

    if fuzz.partial_ratio(e, b) < threshold:
        return -1

    # Find position of shorter string within longer string
    longer, shorter = (e, b) if len(e) >= len(b) else (b, e)
    pos = longer.find(shorter)
    if pos == -1:
        # partial_ratio matched but exact substring not found (fuzzy match)
        # use normalised position estimate based on partial_ratio alignment
        return 0.5
    return pos / len(longer)   # 0 = start, 1 = end


def is_substring_match(etl_row, bm_row, threshold):
    return substring_position(etl_row, bm_row, threshold) >= 0

def map_questions(etl_df, bm_df, threshold):
    """
    Match questions within each notegroupID by weighted similarity.
    Fields: question_content (98%), main_indicator (1%), followed_question_content (1%).
    Null fields are excluded from the weighted average.
    Returns:
        matched  : list of (etl_idx, bm_idx, score)
        etl_only : list of etl_idx  → FP rows
        bm_only  : list of bm_idx   → FN rows
    """
    matched  = []
    etl_only = []
    bm_only  = []

    all_ng_ids = sorted(set(etl_df["notegroupID"].unique()) | set(bm_df["notegroupID"].unique()))

    for ng_id in all_ng_ids:
        etl_ng = etl_df[etl_df["notegroupID"] == ng_id]
        bm_ng  = bm_df[bm_df["notegroupID"]  == ng_id]

        if bm_ng.empty:
            etl_only.extend(etl_ng.index.tolist())
            continue
        if etl_ng.empty:
            bm_only.extend(bm_ng.index.tolist())
            continue

        scores = {}
        for ei in etl_ng.index:
            for bi in bm_ng.index:
                primary_score = question_pair_score(etl_ng.loc[ei], bm_ng.loc[bi])
                # Secondary condition: substring match
                if primary_score < threshold and is_substring_match(etl_ng.loc[ei], bm_ng.loc[bi], threshold):
                    pos = substring_position(etl_ng.loc[ei], bm_ng.loc[bi], threshold)
                    # score = threshold + small position bonus (max +1), so later substrings rank higher
                    primary_score = threshold + pos
                scores[(ei, bi)] = primary_score

        used_etl = set()
        used_bm  = set()
        for (ei, bi), score in sorted(scores.items(), key=lambda x: -x[1]):
            if score < threshold:
                break
            if ei in used_etl or bi in used_bm:
                continue
            matched.append((ei, bi, round(score, 2)))
            used_etl.add(ei)
            used_bm.add(bi)

        etl_only.extend([i for i in etl_ng.index if i not in used_etl])
        bm_only.extend( [i for i in bm_ng.index  if i not in used_bm])

    return matched, etl_only, bm_only


matched, etl_only, bm_only = map_questions(etl, bm, MATCH_THRESHOLD)

print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

Matched pairs : 415
ETL-only (FP) : 1
BM-only  (FN) : 15


In [5]:
match_rows = []
for ei, bi, score in matched:
    match_rows.append({
        "notegroupID":    etl.loc[ei, "notegroupID"],
        "etl_questionID": etl.loc[ei, "questionID"],
        "bm_questionID":  bm.loc[bi, "questionID"] if "questionID" in bm.columns else None,
        "etl_question":   etl.loc[ei, "question_content"],
        "bm_question":    bm.loc[bi, "question_content"] if "question_content" in bm.columns else None,
        "score":          score,
    })

#pd.DataFrame(match_rows)

In [6]:
print("=== ETL-only (FP) ===")
display(etl.loc[etl_only, ["notegroupID", "questionID", "question_content"]])

print("\n=== BM-only (FN) ===")
bm_cols = [c for c in ["notegroupID", "questionID", "question_content"] if c in bm.columns]
display(bm.loc[bm_only, bm_cols])

=== ETL-only (FP) ===


,notegroupID,questionID,question_content
366,22,384,Hoe makkelijk of moeilijk was het om de juiste...



=== BM-only (FN) ===


,notegroupID,questionID,question_content
46,3,47,What were the strong elements that helped you...
54,4,55,If you currently have a job:\n ● Where do you ...
82,5,86,Are you looking for new work?
96,5,102,What were the strong elements that helped you ...
98,5,104,"What did you miss, what would have helped you ..."
261,12,279,What do you need to find work?
262,12,280,How can the municipality help you to find a job?
272,13,290,Collecting feedback
289,15,303,Do you see that improved health of people has ...
290,15,304,Do you see that people can be more active in V...


In [7]:
csv_rows = []
for ei, bi, score in matched:
    csv_rows.append({
        "notegroupID":        etl.loc[ei, "notegroupID"],
        "etl_questionID":     etl.loc[ei, "questionID"],
        "bm_questionID":      bm.loc[bi, "questionID"] if "questionID" in bm.columns else None,
        "etl_question_content": etl.loc[ei, "question_content"],
        "bm_question_content":  bm.loc[bi, "question_content"] if "question_content" in bm.columns else None,
        "match_score":        score,
    })

matched_csv = pd.DataFrame(csv_rows)
matched_csv.to_csv(MATCHED_CSV_PATH, index=False)
print(f"Saved {len(matched_csv)} matched pairs to {MATCHED_CSV_PATH}")

Saved 415 matched pairs to matched_questions.csv
